# Pitch scores before and after coaching: sign test

> **Before you run:** choose **Copy to Drive** at the top of this page (or **File → Save a copy in Drive**). Your copy opens in a new tab. In your copy, choose **Runtime → Run all**. Your copy is yours, so Colab runs it without a warning, and it keeps your changes.

**Sign Test** · a worked example from the StatNex teaching guide

**The story.** 20 interns gave a sales pitch, got coaching, and pitched again. One intern was absent for the first round, so that pair cannot be compared.

**The question.** Did coaching improve the interns' pitch scores?

If you run this page without a copy, Colab first warns that the notebook was not written by Google. It reads none of your data, so **Run anyway** is safe, but your changes are not kept.

The first cells save the table in this session. The rest is the calculation StatNex runs, written out in Python. It does not use StatNex, and every number it prints was checked against the app's.

## The table

This cell saves the table `06_nonparametric/sign_test.csv` (20 rows) as `data.csv`, the file the code reads.

In [ ]:
%%writefile data.csv
Intern No.,Before coaching,After coaching
1.0,70,90.0
2.0,56,74.0
3.0,72,93.0
4.0,89,100.0
5.0,57,71.0
6.0,48,52.0
7.0,61,61.0
8.0,72,72.0
9.0,65,60.0
10.0,59,70.0
11.0,55,43.0
12.0,66,68.0
13.0,46,56.0
14.0,54,65.0
15.0,59,64.0
16.0,53,51.0
17.0,Absent,55.0
18.0,73,77.0
19.0,72,93.0
20.0,68,77.0

## Libraries

In [ ]:
import numpy as np
import pandas as pd
from scipy import stats

pd.set_option("display.width", 120)


def stars(p):
    """* for P < 0.05, ** for P < 0.01."""
    return "**" if p < 0.01 else "*" if p < 0.05 else ""

## The data

In [ ]:
# One row per pair; d = variable 1 - variable 2. A pair missing either value is
# dropped. Every column of variable 1 is paired with every column of variable 2.
data = pd.read_csv("data.csv", keep_default_na=False, na_values=[""])
variables1 = ["Before coaching"]
variables2 = ["After coaching"]

sizes, results = {}, []
for a in variables1:
    for b in variables2:
        pairs = data[[a, b]].apply(pd.to_numeric, errors="coerce").dropna()
        d = pairs[a].to_numpy(float) - pairs[b].to_numpy(float)
        sizes.setdefault(a, len(d))
        sizes.setdefault(b, len(d))
        zero, negative, positive = int((d == 0).sum()), int((d < 0).sum()), int((d > 0).sum())
        used = negative + positive
        cases = pd.Series({"d = 0": zero, "d < 0": negative, "d > 0": positive, "total": len(d)},
                          name="pairs")

        # --- Sign test (exact binomial) ---
        # Pairs with d = 0 carry no direction and are left out: report how many pairs
        # the test really used. If nothing had changed (the null hypothesis), each remaining
        # pair would be a fair coin toss, so the number of positive signs is
        # Binomial(used, 1/2), and P is the chance of a split at least this lopsided.
        p_exact = stats.binomtest(positive, used, 0.5).pvalue

        # --- Normal approximation, without and with continuity correction ---
        # z measures how far the smaller sign count lies from used / 2; the correction
        # moves it half a step towards the middle.
        smaller = min(negative, positive)
        z_plain = abs(smaller - used / 2) / (np.sqrt(used) / 2)
        z_corrected = max(0.0, (abs(smaller - used / 2) - 0.5) / (np.sqrt(used) / 2))
        rows = []
        for correction, z in (("none", z_plain), ("applied", z_corrected)):
            p = 2 * stats.norm.sf(z)
            rows.append([correction, z, p, stars(p)])
        normal = pd.DataFrame(rows, columns=["continuity correction", "z", "P", ""])

        results.append({"cases": cases, "p_exact": p_exact, "normal": normal})
        print(f"\n=== d = {a} - {b} ===")
        print(cases.to_string())
        print(f"\nExact binomial: P = {p_exact:.6g} {stars(p_exact)}")
        print(normal.to_string(index=False))

print()
print(pd.Series(sizes, name="n"))

## What the result says

The sign test looks only at the direction of each change, not its size. Of the 20 interns, 19 have both scores. The case summary shows that 14 scored higher after coaching, 3 scored lower and 2 did not change. The test leaves out the 2 unchanged pairs.

If coaching made no difference, rises and falls would be about equally common. The two-sided P value is 0.013. The P value is the chance of a split at least as uneven as 14 against 3 if rises and falls were equally likely. The normal approximation with continuity correction agrees: z = 2.43, P = 0.015.

So more interns improved than chance would explain. But every intern also pitched a second time, and practice alone can raise a score. This design cannot show that the coaching itself caused the rise.

## Your own data

Open the **Files** panel (the folder icon on the left) and upload a CSV file. Then change `"data.csv"` in the `read_csv` line to its name, change the column names in the code to yours, and choose **Run all** again.